# Allen CT — Cell 565871768

**Dendrite type:** aspiny  |  **Area:** VISp  |  **Layer:** 5  |  **Line:** Oxtr-T2A-Cre

Per-cell analysis: spike fit visualization, waveform gallery, feature distributions by sweep and stimulus amplitude.

In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── adjust these two paths if opening from a different working directory ──────
_here = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_here, '../../allen_ct_helper_modules'))
sys.path.insert(0, os.path.join(_here, '../../../..'))

from data_loader import load_cell_metadata, get_all_spiking_sweeps, load_voltage_trace
from config import ALLEN_CT_PICKLE_ROOT
from spikeparam.patch.fit import Spike

import warnings
warnings.filterwarnings('ignore')

COL_SPINY  = '#CC44CC'
COL_ASPINY = '#00CCCC'
COL_SPARSE = '#AAAAAA'
DEND_COLS  = {'spiny': COL_SPINY, 'aspiny': COL_ASPINY, 'sparsely spiny': COL_SPARSE}
STIM_COLS  = {
    'Long Square':                '#2166AC',
    'Short Square':               '#D53E4F',
    'Ramp':                       '#F46D43',
    'Noise 1':                    '#66C2A5',
    'Noise 2':                    '#3288BD',
    'Square - 2s Suprathreshold': '#ABDDA4',
    'Short Square - Triple':      '#FDAE61',
}
FS_TITLE, FS_LABEL, FS_TICK, FS_ANNOT = 11, 10, 9, 8
FEAT_DIR = os.path.join(ALLEN_CT_PICKLE_ROOT, 'features')
SPIKE_FEAT_COLS = [
    'ramp_amp', 'inflection_time', 'inflection_amp',
    'peak_amp', 'peak_width', 'peak_sharpness',
    'exp_lambda', 'exp_const', 'log_isi',
]
FEAT_LABELS = {
    'ramp_amp':        'Ramp amp (mV/ms)',
    'inflection_time': 'Threshold time (ms)',
    'inflection_amp':  'Threshold voltage (mV)',
    'peak_amp':        'Peak amplitude (mV)',
    'peak_width':      'Spike width (ms)',
    'peak_sharpness':  'Peak sharpness',
    'exp_lambda':      'Repolarization rate λ',
    'exp_const':       'Repolarization const (mV)',
    'log_isi':         'Log ISI',
}

In [ ]:
SPECIMEN_ID = 565871768

## 1. Cell metadata

In [ ]:
cells_df = load_cell_metadata(species='Mus musculus')
row = cells_df[cells_df['id'] == SPECIMEN_ID].iloc[0]
display(row.to_frame().T)

## 2. Spike fit — example Long Square sweep

`sp.plot(show_points=True)` shows the aligned waveform gallery, mean waveform, and annotated landmark points + fitted exponential decay.

In [ ]:
def _best_long_square(sid):
    sweeps = get_all_spiking_sweeps(sid)
    ls = [s for s in sweeps if 'Long Square' in s.get('stimulus_name', '')]
    s  = max(ls or sweeps, key=lambda x: x.get('num_spikes') or 0)
    v, _, _, fs, idx = load_voltage_trace(sid, s)
    if idx is not None:
        v = v[idx[0]:idx[1]]
    sp = Spike(thresh_amp=-10, window_length=(5., 5.), smooth_frac=0.008, pre_inflection_ms=0.5)
    sp.fit(list(v), fs, n_jobs=-1)
    sp.filter_features()
    return sp, s

sp, sweep_meta = _best_long_square(SPECIMEN_ID)
print(f"Sweep {sweep_meta['sweep_number']}  |  "
      f"{sweep_meta.get('stimulus_name')}  |  "
      f"{sweep_meta.get('stimulus_absolute_amplitude', 0):.0f} pA  |  "
      f"{len(sp.df_features)} spikes")
sp.plot(show_points=True)
plt.suptitle(f"Specimen {SPECIMEN_ID} — {row['dendrite_type']}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Waveform gallery — all spiking sweeps

Individual spike waveforms (grey) + mean (coloured) for every spiking sweep this cell has.

In [ ]:
feat_path = os.path.join(FEAT_DIR, f'{SPECIMEN_ID}_features.pkl')
wav_path  = os.path.join(FEAT_DIR, f'{SPECIMEN_ID}_waveforms.pkl')

if not os.path.exists(feat_path):
    print(f'Pickle not found — run:  python scripts/run_allen_ct_batch.py --cells {SPECIMEN_ID}')
else:
    cell_spikes = pd.read_pickle(feat_path)
    wav_list    = pickle.load(open(wav_path, 'rb'))   # list of per-sweep waveform arrays

    fig, ax = plt.subplots(figsize=(8, 5))
    for wav_arr in wav_list:
        if wav_arr is None:
            continue
        try:
            wavs = np.array(wav_arr)          # (n_spikes, n_samples)
            if wavs.ndim != 2 or wavs.shape[0] == 0:
                continue
            t_ms = np.linspace(-5, 5, wavs.shape[1])
            for w in wavs:
                ax.plot(t_ms, w, color='#00CCCC', alpha=0.12, linewidth=0.5)
        except Exception:
            continue

    # Compute grand mean from sp (best Long Square sweep, already fitted above)
    try:
        grand_mean = np.array(sp.waveforms)
        t_ms = np.linspace(-5, 5, grand_mean.shape[1])
        ax.plot(t_ms, grand_mean.mean(axis=0), color='#00CCCC', linewidth=2.5,
                label=f'mean (Long Sq, n={len(grand_mean)} spikes)', zorder=5)
    except Exception:
        pass

    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Time from peak (ms)', fontsize=FS_LABEL)
    ax.set_ylabel('Voltage (mV)', fontsize=FS_LABEL)
    ax.set_title(f'Specimen {SPECIMEN_ID} — waveform gallery (all spiking sweeps)',
                 fontsize=FS_TITLE, fontweight='bold', loc='left')
    ax.legend(fontsize=FS_ANNOT, frameon=False)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

## 4. Feature distributions by stimulus type

Violin + strip of spikeparam features across stimulus types for this cell.

In [ ]:
if not os.path.exists(feat_path):
    print('Run batch first.')
else:
    # Load sweep table to add stimulus metadata
    sweep_table = pd.read_pickle(os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_sweep_table.pkl'))
    cell_full = cell_spikes.merge(
        sweep_table[['sweep_uid', 'stimulus_name', 'stimulus_absolute_amplitude']],
        on='sweep_uid', how='left'
    )
    FEAT_COLS = [c for c in SPIKE_FEAT_COLS if c in cell_full.columns]

    stim_types = cell_full['stimulus_name'].value_counts().index.tolist()
    n_cols, n_rows = 3, int(np.ceil(len(FEAT_COLS) / 3))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
    axes_flat = list(axes.flat) if n_rows > 1 else list(axes)
    rng = np.random.default_rng(42)

    for ax, feat in zip(axes_flat, FEAT_COLS):
        for i, stim in enumerate(stim_types):
            vals = cell_full[cell_full['stimulus_name'] == stim][feat].dropna().values
            if len(vals) < 2:
                continue
            c = STIM_COLS.get(stim, '#888888')
            parts = ax.violinplot(vals, positions=[i], widths=0.65,
                                  showmedians=False, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(c); pc.set_alpha(0.4); pc.set_linewidth(0.5)
            jit = rng.uniform(-0.1, 0.1, len(vals))
            ax.scatter(i + jit, vals, color=c, s=5, alpha=0.6, linewidths=0, zorder=3)
            ax.plot([i - 0.15, i + 0.15], [np.median(vals)] * 2, 'k-', lw=2, zorder=5)
        ax.set_xticks(range(len(stim_types)))
        ax.set_xticklabels([s.replace(' ', '\n') for s in stim_types], fontsize=6)
        ax.set_ylabel(FEAT_LABELS.get(feat, feat), fontsize=FS_LABEL)
        ax.set_title(FEAT_LABELS.get(feat, feat), fontsize=FS_TITLE,
                     fontweight='bold', loc='left')
        sns.despine(ax=ax)
    for ax in axes_flat[len(FEAT_COLS):]:
        ax.set_visible(False)
    fig.suptitle(f'Specimen {SPECIMEN_ID} — features by stimulus type',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 5. Feature variation with stimulus amplitude (Long Square)

How do waveform features change as we increase the current injection amplitude? Each point is one spike; line = median per amplitude level.

In [ ]:
if not os.path.exists(feat_path):
    print('Run batch first.')
else:
    ls_cell = cell_full[cell_full['stimulus_name'].str.contains('Long Square', na=False)]
    amps = np.sort(ls_cell['stimulus_absolute_amplitude'].unique())

    show = [f for f in ['peak_amp', 'peak_width', 'exp_lambda', 'inflection_amp'] if f in FEAT_COLS]
    fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 4))
    if len(show) == 1:
        axes = [axes]
    c = DEND_COLS.get(row['dendrite_type'], '#888888')
    rng3 = np.random.default_rng(0)
    for ax, feat in zip(axes, show):
        for a in amps:
            vals = ls_cell[ls_cell['stimulus_absolute_amplitude'] == a][feat].dropna().values
            if len(vals) == 0:
                continue
            jit = rng3.uniform(-3, 3, len(vals))
            ax.scatter(a + jit, vals, color=c, s=10, alpha=0.5, linewidths=0)
            ax.plot([a - 8, a + 8], [np.median(vals)] * 2, color=c, lw=2)
        ax.set_xlabel('Stimulus amplitude (pA)', fontsize=FS_LABEL)
        ax.set_ylabel(FEAT_LABELS.get(feat, feat), fontsize=FS_LABEL)
        ax.set_title(FEAT_LABELS.get(feat, feat), fontsize=FS_TITLE,
                     fontweight='bold', loc='left')
        sns.despine(ax=ax)
    fig.suptitle(f'Specimen {SPECIMEN_ID} — feature vs. Long Square amplitude',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()